<a href="https://colab.research.google.com/github/KeshavFTW099/ArtificalAdvanced_NeuralNetworks/blob/main/CaseStudyANN_99.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Building a CNN Classification model (Customized and Pretrained) to identify the Skin cancer disease. Dataset used here is from (Kaggle) **

In [1]:
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

!pip install -q --upgrade kaggle
!kaggle datasets download -d hasnainjaved/melanoma-skin-cancer-dataset-of-10000-images
!unzip -q -o melanoma-skin-cancer-dataset-of-10000-images.zip -d data
!find data -maxdepth 3 -type d

Dataset URL: https://www.kaggle.com/datasets/hasnainjaved/melanoma-skin-cancer-dataset-of-10000-images
License(s): CC0-1.0
100% 98.7M/98.7M [00:00<00:00, 122MB/s]

data
data/melanoma_cancer_dataset
data/melanoma_cancer_dataset/train
data/melanoma_cancer_dataset/train/malignant
data/melanoma_cancer_dataset/train/benign
data/melanoma_cancer_dataset/test
data/melanoma_cancer_dataset/test/malignant
data/melanoma_cancer_dataset/test/benign


In [2]:
import os, glob, numpy as np, tensorflow as tf
from tensorflow.keras import layers, models, regularizers

train_dir = os.path.dirname(glob.glob("data/**/train/benign", recursive=True)[0])
test_dir  = os.path.join(os.path.dirname(train_dir), "test")

IMG, BATCH, SEED = (224, 224), 32, 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.15, subset="training", seed=SEED,
    image_size=IMG, batch_size=BATCH, label_mode="binary")
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.15, subset="validation", seed=SEED,
    image_size=IMG, batch_size=BATCH, label_mode="binary")
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=IMG, batch_size=BATCH, label_mode="binary", shuffle=False)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.prefetch(tf.data.AUTOTUNE)

Found 9605 files belonging to 2 classes.
Using 8165 files for training.
Found 9605 files belonging to 2 classes.
Using 1440 files for validation.
Found 1000 files belonging to 2 classes.


CUSTOM CNN

In [3]:
def build_cnn(optimizer, dropout=0.5, l2=1e-4):
    model = models.Sequential([
        layers.Input(shape=IMG + (3,)),
        layers.Rescaling(1./255),
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),

        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l2(l2)),   # L2 regularization
        layers.Dropout(dropout),                                # Dropout
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model

cnn = build_cnn(tf.keras.optimizers.Adam(1e-3))
cnn.summary()
cnn_hist = cnn.fit(train_ds, validation_data=val_ds, epochs=10)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
256/256 ━━━━━━━━━━━━━━━━━━━━ 22s 61ms/step - accuracy: 0.7270 - loss: 0.5645 - val_accuracy: 0.8243 - val_loss: 0.4185
Epoch 2/10
256/256 ━━━━━━━━━━━━━━━━━━━━ 15s 60ms/step - accuracy: 0.8340 - loss: 0.4050 - val_accuracy: 0.8479 - val_loss: 0.3341
Epoch 3/10
256/256 ━━━━━━━━━━━━━━━━━━━━ 21s 60ms/step - accuracy: 0.8601 - loss: 0.3566 - val_accuracy: 0.8556 - val_loss: 0.3496
Epoch 4/10
256/256 ━━━━━━━━━━━━━━━━━━━━ 16s 62ms/step - accuracy: 0.8638 - loss: 0.3469 - val_accuracy: 0.8764 - val_loss: 0.3174
Epoch 5/10
256/256 ━━━━━━━━━━━━━━━━━━━━ 16s 61ms/step - accuracy: 0.8596 - loss: 0.3534 - val_accuracy: 0.8521 - val_loss: 0.3601
Epoch 6/10
256/256 ━━━━━━━━━━━━━━━━━━━━ 16s 61ms/step - accuracy: 0.8679 - loss: 0.3393 - val_accuracy: 0.8715 - val_loss: 0.3426
Epoch 7/10
256/256 ━━━━━━━━━━━━━━━━━━━━ 21s 62ms/step - accuracy: 0.8647 - loss: 0.3390 - val_accuracy: 0.8708 - val_loss: 0.3061
Epoch 8/10
256/256 ━━━━━━━━━━━━━━━━━━━━ 16s 61ms/step - accuracy: 0.8723 - loss: 0.3285 - 

USING PRETRAINED

In [4]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

base = ResNet50(include_top=False, weights="imagenet", input_shape=IMG + (3,))
base.trainable = False

inputs = layers.Input(shape=IMG + (3,))
x = layers.RandomFlip("horizontal_and_vertical")(inputs)
x = layers.RandomRotation(0.2)(x)
x = layers.Lambda(preprocess_input)(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

resnet = models.Model(inputs, outputs)
resnet.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
               loss="binary_crossentropy", metrics=["accuracy"])

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2),
]

# Phase 1: train only the new head
h1 = resnet.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=callbacks)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/8
256/256 ━━━━━━━━━━━━━━━━━━━━ 52s 179ms/step - accuracy: 0.8570 - loss: 0.3590 - val_accuracy: 0.9056 - val_loss: 0.2470 - learning_rate: 0.0010
Epoch 2/8
256/256 ━━━━━━━━━━━━━━━━━━━━ 43s 166ms/step - accuracy: 0.8927 - loss: 0.2874 - val_accuracy: 0.9118 - val_loss: 0.2340 - learning_rate: 0.0010
Epoch 3/8
256/256 ━━━━━━━━━━━━━━━━━━━━ 43s 168ms/step - accuracy: 0.8976 - loss: 0.2733 - val_accuracy: 0.9187 - val_loss: 0.2385 - learning_rate: 0.0010
Epoch 4/8
256/256 ━━━━━━━━━━━━━━━━━━━━ 82s 170ms/step - accuracy: 0.8981 - loss: 0.2635 - val_accuracy: 0.9146 - val_loss: 0.2265 - learning_rate: 0.0010
Epoch 5/8
256/256 ━━━━━━━━━━━━━━━━━━━━ 42s 166ms/step - accuracy: 0.9018 - loss: 0.2574 - val_accuracy: 0.9139 - val_loss: 0.2326 - learning_rate: 0.0010
Epoch 6/8
256/256 ━━━━━━━━━━━━━━━━━━━━ 83s 170ms/step - accuracy: 0.9122 - loss: 0.2460 - val_accuracy: 0.8986 - val_loss: 0.2614 - learning_rate: 0.0010
Epoch 7/8
256/256 ━━━━━━━

In [5]:
base.trainable = True
for layer in base.layers[:-100]:         # unfreeze the last ~100 layers
    layer.trainable = False
for layer in base.layers:                # keep BatchNorm frozen
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

resnet.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
               loss="binary_crossentropy", metrics=["accuracy"])

h2 = resnet.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=callbacks)

Epoch 1/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 92s 307ms/step - accuracy: 0.9091 - loss: 0.2489 - val_accuracy: 0.8979 - val_loss: 0.2549 - learning_rate: 2.0000e-05
Epoch 2/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 71s 279ms/step - accuracy: 0.9308 - loss: 0.1986 - val_accuracy: 0.9236 - val_loss: 0.2118 - learning_rate: 2.0000e-05
Epoch 3/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 71s 277ms/step - accuracy: 0.9396 - loss: 0.1923 - val_accuracy: 0.9312 - val_loss: 0.1950 - learning_rate: 2.0000e-05
Epoch 4/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 71s 277ms/step - accuracy: 0.9415 - loss: 0.1740 - val_accuracy: 0.9424 - val_loss: 0.1774 - learning_rate: 2.0000e-05
Epoch 5/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 75s 292ms/step - accuracy: 0.9428 - loss: 0.1661 - val_accuracy: 0.9132 - val_loss: 0.2257 - learning_rate: 2.0000e-05
Epoch 6/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 71s 277ms/step - accuracy: 0.9477 - loss: 0.1541 - val_accuracy: 0.9326 - val_loss: 0.2010 - learning_rate: 2.0000e-05
Epoch 7/20
256/256 ━━━━━━━━━━━━━━━━━━━━ 71s 27

In [12]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as dn_pre

def build_transfer(base_fn, pre_fn, dropout=0.5, l2=1e-3, l1=0.0):
    base = base_fn(include_top=False, weights="imagenet", input_shape=IMG + (3,))
    base.trainable = False
    inp = layers.Input(shape=IMG + (3,))
    x = layers.RandomFlip("horizontal_and_vertical")(inp)
    x = layers.RandomRotation(0.25)(x)
    x = layers.RandomZoom(0.15)(x)
    x = layers.RandomContrast(0.15)(x)
    x = layers.Lambda(pre_fn)(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l1_l2(l1=l1, l2=l2))(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return base, models.Model(inp, out)

def train_transfer(base, model, unfreeze=100, lr_head=1e-3, lr_ft=2e-5):
    cb = [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
          tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2)]
    model.compile(optimizer=tf.keras.optimizers.Adam(lr_head),
                  loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=6, callbacks=cb, verbose=2)

    base.trainable = True
    for l in base.layers[:-unfreeze]:
        l.trainable = False
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(lr_ft),
                  loss="binary_crossentropy", metrics=["accuracy"])
    h = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=cb, verbose=2)
    return max(h.history["val_accuracy"])

In [13]:
base_d, densenet = build_transfer(DenseNet121, dn_pre, dropout=0.5, l2=1e-3)
best_val = train_transfer(base_d, densenet, unfreeze=150)
print("DenseNet121 best val acc:", best_val)
print("DenseNet121 test acc    :", densenet.evaluate(test_ds, verbose=0)[1])

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/6
256/256 - 65s - 254ms/step - accuracy: 0.7868 - loss: 0.5811 - val_accuracy: 0.8549 - val_loss: 0.4154 - learning_rate: 0.0010
Epoch 2/6
256/256 - 47s - 183ms/step - accuracy: 0.8344 - loss: 0.4673 - val_accuracy: 0.8840 - val_loss: 0.3683 - learning_rate: 0.0010
Epoch 3/6
256/256 - 46s - 181ms/step - accuracy: 0.8420 - loss: 0.4450 - val_accuracy: 0.8458 - val_loss: 0.3951 - learning_rate: 0.0010
Epoch 4/6
256/256 - 47s - 182ms/step - accuracy: 0.8453 - loss: 0.4204 - val_accuracy: 0.8597 - val_loss: 0.3554 - learning_rate: 0.0010
Epoch 5/6
256/256 - 47s - 182ms/step - accuracy: 0.8475 - loss: 0.4199 - val_accuracy: 0.8632 - val_loss: 0.3605 - learning_rate: 0.0010
Epoch 6/6
256/256 - 46s - 181ms/step - accuracy: 0.8508 - loss: 0.4084 - val_accuracy: 0.8660 - val_loss: 0.3451 - learning_rate: 0.0010
Epoch 1/20
256/256 - 81s - 317ms/step - accuracy: 0.8670 - loss: 0.3965 - val_accuracy: 0.8743 - val_loss: 0.3564 - learning_ra

In [9]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_true = np.concatenate([y.numpy() for _, y in test_ds]).ravel()
y_prob = resnet.predict(test_ds, verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)

print("Custom CNN test accuracy :", round(cnn.evaluate(test_ds, verbose=0)[1], 4))
print("ResNet50 test accuracy   :", round((y_pred == y_true).mean(), 4))
print("ResNet50 ROC-AUC         :", round(roc_auc_score(y_true, y_prob), 4))
print("\nConfusion matrix:\n", confusion_matrix(y_true, y_pred))
print("\n", classification_report(y_true, y_pred, target_names=["benign", "malignant"], digits=4))

Custom CNN test accuracy : 0.879
ResNet50 test accuracy   : 0.915
ResNet50 ROC-AUC         : 0.9775

Confusion matrix:
 [[482  18]
 [ 67 433]]

               precision    recall  f1-score   support

      benign     0.8780    0.9640    0.9190       500
   malignant     0.9601    0.8660    0.9106       500

    accuracy                         0.9150      1000
   macro avg     0.9190    0.9150    0.9148      1000
weighted avg     0.9190    0.9150    0.9148      1000



In [10]:
test_acc  = resnet.evaluate(test_ds,  verbose=0)[1]
train_acc = resnet.evaluate(train_ds, verbose=0)[1]

print(f"ResNet50 Test accuracy  : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"ResNet50 Train accuracy : {train_acc:.4f}  ({train_acc*100:.2f}%)")

ResNet50 Test accuracy  : 0.9150  (91.50%)
ResNet50 Train accuracy : 0.9671  (96.71%)
